# 03 - Align the AIGCodeSet embeddings with our splits
The embeddings CSV has its own row order. We match each of our samples to its embedding row by code (ignoring whitespace), then save one array per split.
**Run the first two cells, look at the columns, and only then change the config if the auto-detection picked the wrong columns.**

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))
from paths import *
print("repo root:", ROOT)

In [ ]:
import ast, hashlib, json, re
import numpy as np
import pandas as pd

EMB_CSV = find_embeddings_csv()
assert EMB_CSV is not None, "Embeddings csv not found in data/external/aigcodeset"
head = pd.read_csv(EMB_CSV, nrows=3)
print("columns:", head.columns.tolist())
display(head.head(2).T.head(25))

In [ ]:
# ---- CONFIG: leave as None for auto-detect, or type the column names you see above ----
CODE_COL_CSV = None   # column with the code text, e.g. "code"
EMB_COLS = None       # None = auto. Or a list of column names, or one column name holding the whole vector

if CODE_COL_CSV is None:
    for c in ["code", "Code", "code_raw", "source_code", "submission_code"]:
        if c in head.columns:
            CODE_COL_CSV = c
            break
assert CODE_COL_CSV, "Could not find the code column, set CODE_COL_CSV by hand"

if EMB_COLS is None:
    cand = [c for c in head.columns if "emb" in str(c).lower()]
    EMB_COLS = cand if cand else None
if isinstance(EMB_COLS, str):
    EMB_COLS = [EMB_COLS]
assert EMB_COLS, "Could not find the embedding column(s), set EMB_COLS by hand"
print("code column:", CODE_COL_CSV)
print("embedding column(s):", EMB_COLS[:5], "..." if len(EMB_COLS) > 5 else "")

In [ ]:
def key(code):
    return hashlib.md5(re.sub(r"\s+", "", str(code)).encode("utf-8")).hexdigest()

def parse_vec(v):
    if isinstance(v, (list, tuple, np.ndarray)):
        return np.asarray(v, dtype=np.float32)
    s = str(v).strip()
    try:
        return np.asarray(json.loads(s), dtype=np.float32)
    except Exception:
        return np.asarray(ast.literal_eval(s), dtype=np.float32)

emb = pd.read_csv(EMB_CSV, usecols=[CODE_COL_CSV] + EMB_COLS)
emb["_key"] = emb[CODE_COL_CSV].map(key)
print("embedding rows:", len(emb), "| duplicate keys:", int(emb["_key"].duplicated().sum()))
emb = emb.drop_duplicates("_key").reset_index(drop=True)
key_to_row = {k: i for i, k in enumerate(emb["_key"])}

def row_vector(i):
    if len(EMB_COLS) == 1:
        return parse_vec(emb.loc[i, EMB_COLS[0]])
    return emb.loc[i, EMB_COLS].to_numpy(dtype=np.float32)

DIM = row_vector(0).shape[0]
print("embedding dimension:", DIM)

In [ ]:
summary = []
for split in ["train", "val", "test"]:
    df = pd.read_parquet(PROCESSED / f"{split}.parquet")
    cc = code_col(df)
    y = to_binary_label(df[LABEL])
    rows = df[cc].map(key).map(key_to_row)
    has = rows.notna().to_numpy()
    arr = np.zeros((len(df), DIM), dtype=np.float32)
    for pos in np.flatnonzero(has):
        arr[pos] = row_vector(int(rows.iloc[pos]))
    np.save(FEATURES / f"emb_{split}.npy", arr)
    np.save(FEATURES / f"emb_{split}_has.npy", has)
    summary.append({"split": split, "samples": len(df), "matched": int(has.sum()),
                    "matched_%": round(100 * has.mean(), 1),
                    "human_matched": int((has & (y == 0)).sum()), "ai_matched": int((has & (y == 1)).sum())})
summary = pd.DataFrame(summary)
display(summary)
print("Total samples without an embedding:", int((summary["samples"] - summary["matched"]).sum()))

If `matched_%` is far below about 95%, do not continue. Send the output of the cells above (columns and the summary table). Phase 1 cleaning removes some rows and strips generation leftovers, so a few unmatched samples are normal.